In [ ]:
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tensorflow.keras.models import clone_model
from tensorflow.keras.optimizers import Adam
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import os
import pandas as pd
from tensorflow.keras.models import load_model
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Dropout
from tensorflow.keras.regularizers import l2

In [ ]:
# Cargar imágenes sin binarizar, con división por clase
X_train_clean = np.load("../data/X_train_class.npy")
X_test_clean = np.load("../data/X_test_class.npy")

In [ ]:
import numpy as np
import os

# Directorio donde guardaste los archivos
ruido_dir = "../data"

# Factores de ruido que generaste
noise_factors = [0.2, 0.39, 0.6]

# Diccionarios para almacenar los datasets
train_noisy = {}
test_noisy = {}

# Cargar datasets ruidosos
for nf in noise_factors:
    nf_str = str(nf).replace(".", "")
    print(f"Cargando datasets con factor de ruido = {nf}")

    # Cargar train y test ruidosos
    X_train_noisy = np.load(os.path.join(ruido_dir, f"train_gauss_{nf_str}.npy"))
    X_test_noisy = np.load(os.path.join(ruido_dir, f"test_gauss_{nf_str}.npy"))

    # Guardar en diccionario
    train_noisy[nf] = X_train_noisy
    test_noisy[nf] = X_test_noisy

    print(f" -> Train: {X_train_noisy.shape}, Test: {X_test_noisy.shape}")

print("\n✅ Datasets ruidosos cargados y listos para usar.")


### Mejor modelo para ruido estructurado reentrenado con ruido gaussiano

In [ ]:
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, UpSampling2D, Dropout
from tensorflow.keras.regularizers import l2

# === Arquitectura del autoencoder pequeño ===
entrada = Input(shape=(64, 128, 1))

# Codificador
x = Conv2D(32, (3, 3), activation='relu', padding='same',
           kernel_regularizer=l2(1e-4))(entrada)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Dropout(0.3)(x)

x = Conv2D(64, (3, 3), activation='relu', padding='same',
           kernel_regularizer=l2(1e-4))(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Dropout(0.3)(x)

# Codificación (representación comprimida)
x = Conv2D(128, (3, 3), activation='relu', padding='same',
           kernel_regularizer=l2(1e-4))(x)

# Decodificador
x = UpSampling2D((2, 2))(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same',
           kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.3)(x)

x = UpSampling2D((2, 2))(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same',
           kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.3)(x)

# Capa de salida
salida = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

# Modelo final
autoencoder_nb = Model(entrada, salida)

# Compilación
autoencoder_nb.compile(optimizer='adam',
                       loss='mean_squared_error',
                       metrics=['mae'])

# Resumen del modelo
autoencoder_nb.summary()


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, UpSampling2D, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

# === Entrenamiento con los 3 factores de ruido ===

# Factores de ruido
noise_factors = [0.2, 0.39, 0.6]

# Directorios
ruido_dir = "../data"
model_dir = "modelos_autoencoder_nb"
os.makedirs(model_dir, exist_ok=True)

# Datos limpios
X_train_clean = np.load(os.path.join(ruido_dir, "X_train_class.npy"))
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Diccionario para guardar resultados
mae_denoise = {}

for nf in noise_factors:
    nf_str = str(nf).replace(".", "")
    print(f"Entrenando autoencoder pequeño con ruido gaussiano: factor = {nf}")

    # Cargar imágenes ruidosas
    X_train_noisy = np.load(os.path.join(ruido_dir, f"train_gauss_{nf_str}.npy"))
    X_test_noisy = np.load(os.path.join(ruido_dir, f"test_gauss_{nf_str}.npy"))

    # Reinstanciar modelo limpio para cada entrenamiento
    modelo_denoise = Model(autoencoder_nb.input, autoencoder_nb.output)
    modelo_denoise.compile(optimizer=Adam(learning_rate=1e-3),
                           loss='mean_squared_error',
                           metrics=['mae'])

    # Entrenar
    historial = modelo_denoise.fit(
        X_train_noisy, X_train_clean,
        epochs=50,
        batch_size=32,
        shuffle=True,
        validation_data=(X_test_noisy, X_test_clean),
        verbose=0
    )

    # Guardar modelo
    model_name = f"autoencoder_nb_gauss_{nf_str}.keras"
    modelo_denoise.save(os.path.join(model_dir, model_name))

    # Guardar métrica
    mae_val = historial.history['val_mae'][-1]
    mae_denoise[nf] = mae_val
    print(f"✅ MAE de validación (factor={nf}): {mae_val:.4f}")

# === Resumen en tabla y heatmap ===
df_mae = pd.DataFrame.from_dict(mae_denoise, orient='index', columns=['MAE'])
print("\nResumen de MAE por factor de ruido:")
print(df_mae)

plt.figure(figsize=(6, 4))
sns.heatmap(df_mae.T, annot=True, fmt=".4f", cmap="YlGnBu", cbar_kws={'label': 'MAE'})
plt.title("Heatmap del MAE por factor de ruido (Autoencoder NB)")
plt.xlabel("Factor de ruido")
plt.ylabel("Métrica")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import random
from tensorflow.keras.models import load_model

# === Configuración ===
factor_ruido = 0.39
nf_str = str(factor_ruido).replace(".", "")
model_dir = "modelos_autoencoder_nb_gauss"
ruido_dir = "../data"

# === Cargar modelo entrenado para factor 0.39 ===
model_name = f"autoencoder_nb_gauss_{nf_str}.keras"
modelo_cargado = load_model(os.path.join(model_dir, model_name))

# === Cargar datos (original y ruidoso) ===
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))
X_test_noisy = np.load(os.path.join(ruido_dir, f"test_gauss_{nf_str}.npy"))

# === Reconstrucciones ===
decoded_imgs = modelo_cargado.predict(X_test_noisy)

# === Selección aleatoria de 7 imágenes ===
n = 7
indices = random.sample(range(len(X_test_clean)), n)

plt.figure(figsize=(15, 6))

for i, idx in enumerate(indices):
    # Original
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(X_test_clean[idx].squeeze(), cmap="gray")
    plt.title("Original")
    plt.axis("off")
    
    # Ruidosa
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(X_test_noisy[idx].squeeze(), cmap="gray")
    plt.title("Ruidosa")
    plt.axis("off")
    
    # Reconstruida
    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(decoded_imgs[idx].squeeze(), cmap="gray")
    plt.title("Reconstruida")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from tensorflow.keras.models import load_model

# === Factores de ruido ===
noise_factors = [0.2, 0.39, 0.6]

# Directorios
ruido_dir = "../data"
model_dir = "modelos_autoencoder_nb_gauss"

# Cargar datos limpios
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Umbral para contar píxeles negros
umbral = 0.5

# Lista para guardar resultados
resultados = []

for nf in noise_factors:
    nf_str = str(nf).replace(".", "")
    print(f"Procesando factor de ruido = {nf}")

    # Cargar modelo entrenado
    model_name = f"autoencoder_nb_gauss_{nf_str}.keras"
    modelo_cargado = load_model(os.path.join(model_dir, model_name))

    # Cargar imágenes ruidosas
    X_test_noisy = np.load(os.path.join(ruido_dir, f"test_gauss_{nf_str}.npy"))

    # Reconstrucciones
    decoded_imgs = modelo_cargado.predict(X_test_noisy, verbose=0)

    # Calcular número de píxeles negros en cada imagen original
    num_pixeles_negros = np.sum(X_test_clean < umbral, axis=(1,2,3))

    # Calcular MSE por imagen
    mse_values = np.mean((X_test_clean - decoded_imgs) ** 2, axis=(1,2,3))

    # Guardar resultados
    for px, mse in zip(num_pixeles_negros, mse_values):
        resultados.append([px, mse, nf])

# === Convertir a DataFrame ===
df = pd.DataFrame(resultados, columns=["pixeles_negros", "mse", "factor_ruido"])

# === Graficar scatter multicolor ===
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x="pixeles_negros", y="mse", hue="factor_ruido", alpha=0.6)

plt.xlabel("Número de píxeles negros")
plt.ylabel("MSE")
plt.title("Relación entre cantidad de píxeles negros y error de reconstrucción")
plt.legend(title="Factor de ruido")
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from scipy.stats import spearmanr

# === Factores de ruido ===
noise_factors = [0.2, 0.39, 0.6]

# Directorios
ruido_dir = "../data"
model_dir = "modelos_autoencoder_nb_gauss"

# Cargar datos limpios
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Umbral para contar píxeles negros
umbral = 0.5

# Lista para guardar resultados
resultados = []

for nf in noise_factors:
    nf_str = str(nf).replace(".", "")
    print(f"Procesando factor de ruido = {nf}")

    # Cargar modelo entrenado
    model_name = f"autoencoder_nb_gauss_{nf_str}.keras"
    modelo_cargado = load_model(os.path.join(model_dir, model_name))

    # Cargar imágenes ruidosas
    X_test_noisy = np.load(os.path.join(ruido_dir, f"test_gauss_{nf_str}.npy"))

    # Reconstrucciones
    decoded_imgs = modelo_cargado.predict(X_test_noisy, verbose=0)

    # Calcular número de píxeles negros en cada imagen original
    num_pixeles_negros = np.sum(X_test_clean < umbral, axis=(1,2,3))

    # Calcular MSE por imagen
    mse_values = np.mean((X_test_clean - decoded_imgs) ** 2, axis=(1,2,3))

    # Guardar resultados
    for px, mse in zip(num_pixeles_negros, mse_values):
        resultados.append([px, mse, nf])

# === Convertir a DataFrame ===
df = pd.DataFrame(resultados, columns=["pixeles_negros", "mse", "factor_ruido"])

# === Graficar scatter con regresión por cada factor ===
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x="pixeles_negros", y="mse", hue="factor_ruido", alpha=0.6, palette="mako")

# Dibujar regresiones y mostrar correlaciones
for nf in noise_factors:
    sub_df = df[df["factor_ruido"] == nf]
    sns.regplot(data=sub_df, x="pixeles_negros", y="mse",
                scatter=False, ci=None, line_kws={"lw":2})

    # Calcular correlación Spearman
    rho, p = spearmanr(sub_df["pixeles_negros"], sub_df["mse"])
    plt.text(0.7, 0.9 - 0.05*noise_factors.index(nf),
             f"ρ({nf}) = {rho:.2f}", transform=plt.gca().transAxes,
             fontsize=10, color="black")

plt.xlabel("Número de píxeles negros")
plt.ylabel("MSE")
plt.title("Relación entre cantidad de píxeles negros y error de reconstrucción")
plt.legend(title="Factor de ruido")
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
from scipy.stats import pearsonr

# === Scatter + regresión lineal por factor ===
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x="pixeles_negros", y="mse", hue="factor_ruido", alpha=0.7, palette="dark")

# Dibujar líneas de regresión para cada factor
for nf in df["factor_ruido"].unique():
    subset = df[df["factor_ruido"] == nf]
    sns.regplot(
        x=subset["pixeles_negros"], y=subset["mse"],
        scatter=False, ci=None, label=f"Ruido {nf}",
        line_kws={"lw":2}
    )
    
    # Calcular correlación Pearson
    rho, p = pearsonr(subset["pixeles_negros"], subset["mse"])
    
    # Agregar texto en la esquina superior derecha
    plt.text(
        x=df["pixeles_negros"].max()*0.65,
        y=df["mse"].max()*(0.95 - 0.08*list(df["factor_ruido"].unique()).index(nf)), 
        s=f"ρ({nf}) = {rho:.2f}",
        fontsize=10,
        color="black"
    )

plt.xlabel("Número de píxeles negros")
plt.ylabel("MSE")
plt.title("Relación entre cantidad de píxeles negros y error de reconstrucción")
plt.legend(title="Factor de ruido")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import load_model
from scipy.stats import spearmanr

# === Factores de ruido ===
noise_factors = [0.2, 0.39, 0.6]

# Directorios
ruido_dir = "../data"
model_dir = "modelos_autoencoder_nb_gauss"

# Cargar datos limpios
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Umbral para contar píxeles negros
umbral = 0.5

# Lista para guardar resultados
resultados = []

for nf in noise_factors:
    nf_str = str(nf).replace(".", "")
    print(f"Procesando factor de ruido = {nf}")

    # Cargar modelo entrenado
    model_name = f"autoencoder_nb_gauss_{nf_str}.keras"
    modelo_cargado = load_model(os.path.join(model_dir, model_name))

    # Cargar imágenes ruidosas
    X_test_noisy = np.load(os.path.join(ruido_dir, f"test_gauss_{nf_str}.npy"))

    # Reconstrucciones
    decoded_imgs = modelo_cargado.predict(X_test_noisy, verbose=0)

    # Calcular número de píxeles negros en cada imagen original
    num_pixeles_negros = np.sum(X_test_clean < umbral, axis=(1,2,3))

    # Calcular MSE por imagen
    mse_values = np.mean((X_test_clean - decoded_imgs) ** 2, axis=(1,2,3))

    # Guardar resultados
    for px, mse in zip(num_pixeles_negros, mse_values):
        resultados.append([px, mse, nf])

# === Convertir a DataFrame ===
df = pd.DataFrame(resultados, columns=["pixeles_negros", "mse", "factor_ruido"])

# === Graficar scatter multicolor con líneas ===
plt.figure(figsize=(8,6))

# Paleta de colores consistente
#palette = sns.color_palette("ch:s=-.2,r=.6", len(noise_factors))
palette = sns.color_palette("mako", len(noise_factors))
#palette = sns.color_palette("flare", len(noise_factors))

sns.scatterplot(
    data=df, x="pixeles_negros", y="mse",
    hue="factor_ruido", alpha=0.6, palette=palette
)

# Añadir líneas de regresión por grupo
for i, nf in enumerate(noise_factors):
    subset = df[df["factor_ruido"] == nf]
    sns.regplot(
        x=subset["pixeles_negros"], y=subset["mse"],
        scatter=False, ci=None, color=palette[i], label=f"Ruido {nf}"
    )

    # Calcular correlación Spearman
    rho, p = spearmanr(subset["pixeles_negros"], subset["mse"])

    # Posición para texto
    plt.text(
        0.95, 0.95 - i*0.05,
        f"ρ({nf}) = {rho:.2f}",
        transform=plt.gca().transAxes,
        ha="right", fontsize=10, color="black"
    )

plt.xlabel("Número de píxeles negros")
plt.ylabel("MSE")
plt.title("Relación entre cantidad de píxeles negros y error de reconstrucción")
plt.legend(title="Factor de ruido")
plt.tight_layout()
plt.show()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model

# === Configuración ===
factor_ruido = 0.2
nf_str = str(factor_ruido).replace(".", "")  # para nombre de archivo

# Directorios
ruido_dir = "../data"
model_dir = "modelos_autoencoder_nb"
os.makedirs(model_dir, exist_ok=True)

# Datos limpios
X_train_clean = np.load(os.path.join(ruido_dir, "X_train_class.npy"))
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Datos ruidosos (factor 0.2)
X_train_noisy = np.load(os.path.join(ruido_dir, f"train_gauss_{nf_str}.npy"))
X_test_noisy = np.load(os.path.join(ruido_dir, f"test_gauss_{nf_str}.npy"))

# === Reinstanciar modelo pequeño ===
modelo_denoise = Model(autoencoder_nb.input, autoencoder_nb.output)
modelo_denoise.compile(optimizer=Adam(learning_rate=1e-3),
                       loss='mean_squared_error',
                       metrics=['mae'])

# === Entrenamiento ===
historial = modelo_denoise.fit(
    X_train_noisy, X_train_clean,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_data=(X_test_noisy, X_test_clean),
    verbose=1
)

# === Guardar modelo entrenado ===
model_name = f"autoencoder_nb_gauss_{nf_str}.keras"
modelo_denoise.save(os.path.join(model_dir, model_name))
print(f"✅ Modelo guardado en {os.path.join(model_dir, model_name)}")

# === Recuperar métricas finales ===
loss_final = historial.history['loss'][-1]
val_loss_final = historial.history['val_loss'][-1]
mae_final = historial.history['mae'][-1]
val_mae_final = historial.history['val_mae'][-1]

print(f"📊 Resultados finales (factor={factor_ruido}):")
print(f"Loss entrenamiento: {loss_final:.4f}")
print(f"Loss validación:    {val_loss_final:.4f}")
print(f"MAE entrenamiento:  {mae_final:.4f}")
print(f"MAE validación:     {val_mae_final:.4f}")

# === Graficar curvas de pérdida y MAE ===
epochs = range(1, len(historial.history['loss']) + 1)

plt.figure(figsize=(12, 5))

# Curva de Pérdida
plt.subplot(1, 2, 1)
plt.plot(epochs, historial.history['loss'], label="Entrenamiento")
plt.plot(epochs, historial.history['val_loss'], label="Validación")
plt.title(f"Curva de Pérdida (factor={factor_ruido})")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()

# Curva de MAE
plt.subplot(1, 2, 2)
plt.plot(epochs, historial.history['mae'], label="Entrenamiento")
plt.plot(epochs, historial.history['val_mae'], label="Validación")
plt.title(f"Curva de MAE (factor={factor_ruido})")
plt.xlabel("Época")
plt.ylabel("MAE")
plt.legend()

plt.tight_layout()
plt.show()



In [ ]:
# === Graficar curva de Pérdida ===
epochs = range(1, len(historial.history['loss']) + 1)

plt.figure(figsize=(6, 5))
plt.plot(epochs, historial.history['loss'], label="Entrenamiento")
plt.plot(epochs, historial.history['val_loss'], label="Validación")
plt.title(f"Curva de Pérdida (factor={factor_ruido})")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.ylim(0.015, 0.04)  # Ajuste del rango (mínimo y máximo)
plt.legend()
plt.tight_layout()
plt.show()

# === Graficar curva de MAE ===
plt.figure(figsize=(6, 5))
plt.plot(epochs, historial.history['mae'], label="Entrenamiento")
plt.plot(epochs, historial.history['val_mae'], label="Validación")
plt.title(f"Curva de MAE (factor={factor_ruido})")
plt.xlabel("Época")
plt.ylabel("MAE")
plt.ylim(0.050, 0.1)  # Ajuste del rango (mínimo y máximo)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# === Configuración ===
noise_factors = [0.2, 0.39, 0.6]
ruido_dir = "../data"
model_dir = "modelos_autoencoder_nb_gauss"

# Cargar datos limpios
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Umbral (opcional, si se requiere calcular pixeles negros)
umbral = 0.5

# Lista de resultados
resultados = []

# Procesar cada factor de ruido
for nf in noise_factors:
    nf_str = str(nf).replace(".", "")
    print(f"Procesando factor de ruido = {nf}")

    # Cargar modelo
    model_name = f"autoencoder_nb_gauss_{nf_str}.keras"
    modelo_cargado = load_model(os.path.join(model_dir, model_name))

    # Cargar datos ruidosos
    X_test_noisy = np.load(os.path.join(ruido_dir, f"test_gauss_{nf_str}.npy"))

    # Reconstrucciones
    decoded_imgs = modelo_cargado.predict(X_test_noisy, verbose=0)

    # Calcular MSE por imagen
    mse_values = np.mean((X_test_clean - decoded_imgs) ** 2, axis=(1,2,3))

    # Guardar resultados en lista
    for mse in mse_values:
        resultados.append([nf, mse])

# Convertir a DataFrame
df_gauss = pd.DataFrame(resultados, columns=["factor_ruido", "mse"])

# === Graficar Boxplot ===
plt.figure(figsize=(8,6))
sns.boxplot(data=df_gauss, x="factor_ruido", y="mse", palette="mako")
plt.xlabel("Factor de ruido gaussiano")
plt.ylabel("MSE por imagen")
plt.title("Distribución de errores de reconstrucción (ruido gaussiano)")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Seleccionar algunas imágenes representativas
indices = [0, 50, 120]  # <- cámbialos según los casos interesantes
fig, axes = plt.subplots(len(indices), 4, figsize=(12, 3*len(indices)))

for i, idx in enumerate(indices):
    original = X_test_clean[idx].squeeze()
    noisy = X_test_noisy[idx].squeeze()
    reconstruida = decoded_imgs[idx].squeeze()
    error_map = (original - reconstruida) ** 2

    # Original
    axes[i,0].imshow(original, cmap="gray")
    axes[i,0].set_title("Original")
    axes[i,0].axis("off")

    # Ruidosa
    axes[i,1].imshow(noisy, cmap="gray")
    axes[i,1].set_title(f"Noisy (factor={factor_ruido})")
    axes[i,1].axis("off")

    # Reconstruida
    axes[i,2].imshow(reconstruida, cmap="gray")
    axes[i,2].set_title("Reconstruida")
    axes[i,2].axis("off")

    # Heatmap de error
    #im = axes[i,3].imshow(error_map, cmap="hot")
    im = axes[i,3].imshow(error_map, cmap="plasma")
    axes[i,3].set_title("Mapa de error")
    axes[i,3].axis("off")
    fig.colorbar(im, ax=axes[i,3], fraction=0.046, pad=0.04)

plt.suptitle("Heatmaps de error por posición (ruido gaussiano)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim

# Factores de ruido
noise_factors = [0.2, 0.39, 0.6]

# Directorios
ruido_dir = "../data"
model_dir = "modelos_autoencoder_nb"
os.makedirs(model_dir, exist_ok=True)

# Datos limpios
X_train_clean = np.load(os.path.join(ruido_dir, "X_train_class.npy"))
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Diccionario para guardar resultados
metrics_denoise = {}

for nf in noise_factors:
    nf_str = str(nf).replace(".", "")
    print(f"\n🚀 Entrenando autoencoder pequeño con ruido gaussiano: factor = {nf}")

    # Cargar imágenes ruidosas
    X_train_noisy = np.load(os.path.join(ruido_dir, f"train_gauss_{nf_str}.npy"))
    X_test_noisy = np.load(os.path.join(ruido_dir, f"test_gauss_{nf_str}.npy"))

    # Reinstanciar modelo limpio para cada entrenamiento
    modelo_denoise = Model(autoencoder_nb.input, autoencoder_nb.output)
    modelo_denoise.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='mean_squared_error',
        metrics=['mae']
    )

    # Entrenar
    historial = modelo_denoise.fit(
        X_train_noisy, X_train_clean,
        epochs=50,
        batch_size=32,
        shuffle=True,
        validation_data=(X_test_noisy, X_test_clean),
        verbose=0
    )

    # Guardar modelo
    #model_name = f"autoencoder_nb_gauss_{nf_str}.keras"
    #modelo_denoise.save(os.path.join(model_dir, model_name))

    # --- Evaluación en el conjunto de prueba ---
    X_pred = modelo_denoise.predict(X_test_noisy, verbose=0)

    # MSE y MAE usando numpy
    mse_val = np.mean((X_test_clean - X_pred) ** 2)
    mae_val = np.mean(np.abs(X_test_clean - X_pred))

    # PSNR y SSIM (en promedio sobre imágenes)
    psnr_vals = []
    ssim_vals = []
    for i in range(len(X_test_clean)):
        img_true = X_test_clean[i].squeeze()
        img_pred = X_pred[i].squeeze()
        psnr_vals.append(psnr(img_true, img_pred, data_range=1.0))
        ssim_vals.append(ssim(img_true, img_pred, data_range=1.0))
    
    psnr_val = np.mean(psnr_vals)
    ssim_val = np.mean(ssim_vals)

    # Guardar métricas
    metrics_denoise[nf] = {
        "MSE": mse_val,
        "MAE": mae_val,
        "PSNR": psnr_val,
        "SSIM": ssim_val
    }

    print(f"✅ Factor {nf} -> MSE: {mse_val:.6f}, MAE: {mae_val:.6f}, PSNR: {psnr_val:.4f}, SSIM: {ssim_val:.4f}")

# === Resumen en DataFrame ===
df_metrics = pd.DataFrame.from_dict(metrics_denoise, orient="index")
print("\nResumen de métricas por factor de ruido:")
print(df_metrics)

# Opcional: mostrar como heatmap de todas las métricas
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
sns.heatmap(df_metrics.T, annot=True, fmt=".4f", cmap="YlGnBu", cbar=False)
plt.title("Resumen de métricas por factor de ruido (Autoencoder NB)")
plt.xlabel("Factor de ruido")
plt.ylabel("Métrica")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# === Umbral para contar píxeles negros ===
umbral = 0.5  # ajusta según tu binarización

# Número de píxeles negros en cada imagen original
num_pixeles_negros = np.sum(X_test_clean < umbral, axis=(1, 2, 3))

# Calcular MSE por imagen (original vs reconstruida)
mse_values = np.mean((X_test_clean - decoded_imgs) ** 2, axis=(1, 2, 3))

# === Crear scatter ===
plt.figure(figsize=(8, 6))
sns.scatterplot(x=num_pixeles_negros, y=mse_values, alpha=0.6)

# Línea de tendencia (regresión lineal simple)
sns.regplot(x=num_pixeles_negros, y=mse_values, scatter=False, ci=None, line_kws={"color": "red"})

plt.xlabel("Número de píxeles negros")
plt.ylabel("MSE")
plt.title("Relación entre cantidad de píxeles negros y error de reconstrucción")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr  # para correlación monotónica

# --- Usa los arrays que ya tienes en memoria ---
# X_test_clean: imágenes limpias
# decoded_imgs: reconstrucciones del modelo para factor 0.39

# Umbral para considerar "negro" (ajústalo si tus imágenes son binarias o grises)
umbral_negro = 0.5

# 1) # de píxeles negros por imagen (en las imágenes limpias)
num_pix_negros = np.sum(X_test_clean < umbral_negro, axis=(1, 2, 3))

# 2) MSE por imagen entre original y reconstruida
mse_por_imagen = np.mean((X_test_clean - decoded_imgs) ** 2, axis=(1, 2, 3))

# 3) Scatter + curva de tendencia
plt.figure(figsize=(8,6))
sns.scatterplot(x=num_pixeles_negros, y=mse_values, alpha=0.6)
sns.regplot(x=num_pixeles_negros, y=mse_values, scatter=False, ci=None, line_kws={"color":"red"})
plt.xlabel("Número de píxeles negros")
plt.ylabel("MSE")
plt.title("Relación entre densidad de píxeles negros y MSE")
plt.show()

# 4) Correlación (opcional)
rho, p = spearmanr(num_pix_negros, mse_por_imagen)
print(f"Correlación Spearman: ρ = {rho:.3f} (p = {p:.3g})")

